# 🏦 Customer Support Handoff Orchestration

## Overview

This notebook demonstrates **Handoff orchestration** for customer support — a multi-agent pattern where agents dynamically transfer control to one another based on context or user request. Each agent can "handoff" the conversation to another agent with appropriate expertise, ensuring the right specialist handles each part of the task.

Internally, the handoff orchestration is implemented using a **mesh topology** where agents are connected directly without an orchestrator. Each agent can decide when to hand off the conversation based on predefined rules or the content of the messages.

> ⚠️ **Note**: Handoff orchestration only supports `Agent` and the agents must support local tools execution.

### 💼 Industry Use Case: Customer Support with Dynamic Agent Routing

A financial services company uses specialized agents for different support areas:
1. **Triage Agent** receives the initial customer inquiry and routes it
2. **Order Agent** handles order tracking and shipping inquiries
3. **Return Agent** manages product return requests
4. **Refund Agent** processes refund requests

### Key Concepts

| Concept | Description |
|---------|-------------|
| **HandoffBuilder** | High-level API for building handoff workflows |
| **Dynamic Routing** | Agents decide which agent handles the next interaction |
| **with_start_agent()** | Defines which agent receives user input first |
| **add_handoff()** | Configures specific handoff relationships between agents |
| **HandoffAgentUserRequest** | Event emitted when an agent needs user input |
| **Context Synchronization** | Full conversation history maintained across handoffs |
| **Autonomous Mode** | Optional mode that bypasses human input requirements |
| **Tool Approval (HITL)** | Human approval for sensitive tool operations |
| **Checkpointing** | Durable workflows that survive process restarts |

### Differences Between Handoff and Agent-as-Tools

| Aspect | Handoff | Agent-as-Tools |
|--------|---------|----------------|
| **Control Flow** | Explicitly passed between agents; no central authority | Primary agent delegates subtasks; control always returns |
| **Task Ownership** | Receiving agent takes full ownership | Primary agent retains overall responsibility |
| **Context** | Full context handed off entirely | Primary agent manages context selectively |

### Architecture

```
Customer Support Request
           ↓
    Triage Agent (routes inquiry)
           ↓
┌──────────────────────────────────┐
│   Dynamic Handoff Routing        │
│  ├── Order Agent (tracking)      │
│  ├── Return Agent (returns)      │
│  └── Refund Agent (refunds)      │
│       ↕ agents can handoff back  │
│         to triage or each other  │
└──────────────────────────────────┘
           ↓
    Resolution & Workflow Complete
```

### What You'll Learn

- How to create specialized agents for different domains
- How to configure handoff rules between agents
- How to build interactive workflows with dynamic agent routing
- How to handle multi-turn conversations with agent switching
- How to implement tool approval for sensitive operations (HITL)
- How to use checkpointing for durable handoff workflows

## Prerequisites

- ✅ Azure AI Foundry project configured
- ✅ Environment variables: `AI_FOUNDRY_PROJECT_ENDPOINT`, `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- ✅ Azure CLI authentication: Run `az login`

## 1️⃣ Install Dependencies and Import Libraries

In [ ]:
# Install required packages (uncomment if needed)
# %pip install agent-framework agent-framework-orchestrations azure-identity python-dotenv

import asyncio
import os
from typing import Annotated
from dotenv import load_dotenv

from azure.identity import AzureCliCredential

from agent_framework import (
    Agent,
    AgentResponseUpdate,
    Content,
    FileCheckpointStorage,
    Message,
    tool,
    WorkflowEvent,
)
from agent_framework.orchestrations import HandoffBuilder, HandoffAgentUserRequest
from agent_framework.foundry import FoundryChatClient

# Load environment variables from .env file
load_dotenv('../../.env')

# Verify environment is loaded
PROJECT_ENDPOINT = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
print(f"✅ Environment loaded: {PROJECT_ENDPOINT is not None and MODEL_DEPLOYMENT is not None}")

✅ Environment loaded: True


## 2️⃣ Define Simulated Tools for Customer Support

Define three tool-decorated functions that the specialist agents will use. Each tool simulates a real customer support operation.

In [ ]:
@tool
def process_refund(order_number: Annotated[str, "Order number to process refund for"]) -> str:
    """Simulated function to process a refund for a given order number."""
    return f"Refund processed successfully for order {order_number}."

@tool
def check_order_status(order_number: Annotated[str, "Order number to check status for"]) -> str:
    """Simulated function to check the status of a given order number."""
    return f"Order {order_number} is currently being processed and will ship in 2 business days."

@tool
def process_return(order_number: Annotated[str, "Order number to process return for"]) -> str:
    """Simulated function to process a return for a given order number."""
    return f"Return initiated successfully for order {order_number}. You will receive return instructions via email."

print("✅ Customer support tools defined: process_refund, check_order_status, process_return")

✅ Customer support tools defined: process_refund, check_order_status, process_return


## 3️⃣ Set Up the Foundry Chat Client

Initialize the `FoundryChatClient` using `AzureCliCredential` for authentication. This client will be used to create all agents.

In [ ]:
# Create shared Foundry Chat Client
chat_client = FoundryChatClient(
    credential=AzureCliCredential(),
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL_DEPLOYMENT,
)

print("✅ Foundry Chat Client created")

✅ Foundry Chat Client created


## 4️⃣ Define Specialized Customer Support Agents

Create domain-specific agents with a triage coordinator for routing.

### Agent Roles

| Agent | Role | Tools |
|-------|------|-------|
| **Triage Agent** | Routes customer inquiries to the right specialist | None |
| **Order Agent** | Handles order tracking and shipping issues | `check_order_status` |
| **Return Agent** | Manages product return requests | `process_return` |
| **Refund Agent** | Processes refund requests | `process_refund` |

In [ ]:
# Create triage/coordinator agent
triage_agent = Agent(
    client=chat_client,
    instructions=(
        "You are frontline support triage at a financial services company. "
        "Route customer issues to the appropriate specialist agents based on the problem described. "
        "Ask clarifying questions if the issue is unclear before routing."
    ),
    description="Triage agent that handles general inquiries and routes to specialists.",
    name="triage_agent",
)

# Refund specialist: Handles refund requests
refund_agent = Agent(
    client=chat_client,
    instructions=(
        "You process refund requests for a financial services company. "
        "Always verify the order number before processing. "
        "Explain the refund timeline and process to the customer."
    ),
    description="Agent that handles refund requests.",
    name="refund_agent",
    tools=[process_refund],
)

# Order/shipping specialist: Resolves delivery issues
order_agent = Agent(
    client=chat_client,
    instructions=(
        "You handle order and shipping inquiries for a financial services company. "
        "Provide clear status updates and estimated delivery timelines."
    ),
    description="Agent that handles order tracking and shipping issues.",
    name="order_agent",
    tools=[check_order_status],
)

# Return specialist: Handles return requests
return_agent = Agent(
    client=chat_client,
    instructions=(
        "You manage product return requests for a financial services company. "
        "Guide customers through the return process step by step. "
        "If a customer also wants a refund after the return, route them to the refund specialist."
    ),
    description="Agent that handles return processing.",
    name="return_agent",
    tools=[process_return],
)

print("✅ Triage Agent created")
print("✅ Refund Agent created")
print("✅ Order Agent created")
print("✅ Return Agent created")

✅ Triage Agent created
✅ Refund Agent created
✅ Order Agent created
✅ Return Agent created


## 5️⃣ Configure Basic Handoff Workflow with HandoffBuilder

By default, all agents can handoff to each other. The `HandoffBuilder` automatically registers handoff tools with each agent.

| Method | Purpose |
|--------|---------|
| `HandoffBuilder()` | Initialize the handoff orchestration builder |
| `.with_start_agent()` | Define which agent receives user input first |
| `termination_condition` | Lambda to determine when the workflow should stop |
| `.build()` | Create the workflow |

In [5]:
print("\n🏦 Building Customer Support Handoff Workflow...")

# Build handoff workflow using HandoffBuilder
# By default, all agents can handoff to each other
workflow = (
    HandoffBuilder(
        name="customer_support_handoff",
        participants=[triage_agent, refund_agent, order_agent, return_agent],
        termination_condition=lambda conversation: len(conversation) > 0 and "welcome" in conversation[-1].text.lower(),
    )
    .with_start_agent(triage_agent)  # Triage receives initial user input
    .build()
)

print("✅ Handoff workflow built - all agents can handoff to each other")
print("   triage_agent is the start agent")


🏦 Building Customer Support Handoff Workflow...
✅ Handoff workflow built - all agents can handoff to each other
   triage_agent is the start agent


## 6️⃣ Configure Advanced Handoff Rules (Custom Routing)

For more advanced routing, you can configure specific handoff relationships between agents using `add_handoff()`.

### Custom Routing Rules
```
triage_agent ──→ order_agent, return_agent  (cannot route directly to refund)
return_agent ──→ refund_agent, triage_agent (returns can escalate to refunds)
order_agent  ──→ triage_agent              (orders route back to triage)
refund_agent ──→ triage_agent              (refunds route back to triage)
```

> **Note:** Even with custom handoff rules, all agents are still connected in a mesh topology for **context synchronization**. The handoff rules only govern which agents can **take over** the conversation next. Only user and agent messages are synchronized — tool-related contents (including handoff tool calls) are not broadcasted.

In [6]:
print("\n🏦 Building Customer Support Handoff Workflow (Custom Routing)...")

# Build handoff workflow with custom routing rules
workflow_custom = (
    HandoffBuilder(
        name="customer_support_handoff_custom",
        participants=[triage_agent, refund_agent, order_agent, return_agent],
        termination_condition=lambda conversation: len(conversation) > 0 and "welcome" in conversation[-1].text.lower(),
    )
    .with_start_agent(triage_agent)
    # Triage cannot route directly to refund agent
    .add_handoff(triage_agent, [order_agent, return_agent])
    # Only the return agent can handoff to refund agent
    .add_handoff(return_agent, [refund_agent, triage_agent])
    # All specialists can handoff back to triage for further routing
    .add_handoff(order_agent, [triage_agent])
    .add_handoff(refund_agent, [triage_agent])
    .build()
)

print("✅ Custom routing handoff workflow built")
print("   triage → order, return")
print("   return → refund, triage")
print("   order  → triage")
print("   refund → triage")


🏦 Building Customer Support Handoff Workflow (Custom Routing)...
✅ Custom routing handoff workflow built
   triage → order, return
   return → refund, triage
   order  → triage
   refund → triage


## 7️⃣ Run Interactive Handoff Agent Interaction

Unlike other orchestrations, handoff is **interactive** because an agent may not decide to handoff after every turn. If an agent doesn't handoff, human input is required to continue the conversation.

### Interaction Flow
```
User Message → Agent Processes →
  ├── Agent calls handoff tool → Next agent takes over
  └── Agent responds directly  → Workflow pauses with RequestInfoEvent
```

When an agent decides not to handoff, the workflow emits a `RequestInfoEvent` with a `HandoffAgentUserRequest` payload containing the agent's most recent messages. Your application then collects the user's input and calls `workflow.send_responses()` to resume.

> **Note:** Each run cell builds a **fresh workflow** to avoid stale state from previous interrupted runs in the notebook.

In [ ]:
# Build a fresh workflow for the interactive demo
workflow_interactive = (
    HandoffBuilder(
        name="interactive_handoff",
        participants=[triage_agent, refund_agent, order_agent, return_agent],
        termination_condition=lambda conversation: len(conversation) > 0 and "welcome" in conversation[-1].text.lower(),
    )
    .with_start_agent(triage_agent)
    .build()
)

print("\U0001f680 Starting interactive handoff workflow...")
print("=" * 60)

# Start workflow with initial user message using streaming
pending_requests: list[WorkflowEvent] = []

async for event in workflow_interactive.run("I need help with my order", stream=True):
    if event.type == "request_info" and isinstance(event.data, HandoffAgentUserRequest):
        pending_requests.append(event)

# Interactive loop: respond to requests
while pending_requests:
    # Display agent messages and collect user input
    responses: dict[str, object] = {}
    exit_requested = False
    for request in pending_requests:
        request_data = request.data
        print(f"\n\U0001f4ac Agent '{request.executor_id}' says:")
        for msg in request_data.agent_response.messages[-3:]:
            if msg.text:
                print(f"   {msg.author_name or msg.role}: {msg.text}")
        print("-" * 40)

        valid = False
        while not valid:
            user_input = input("You (type message or 'exit'): ").strip()
            if user_input:
                valid = True
            else:
                print("\u26a0\ufe0f Please type a message and press Enter.")
        if user_input.lower() == "exit":
            print("\n\u274c Conversation ended by user.")
            responses[request.request_id] = HandoffAgentUserRequest.terminate()
            exit_requested = True
            break
        responses[request.request_id] = HandoffAgentUserRequest.create_response(user_input)

    # Send responses (including terminate) and collect new requests
    pending_requests = []
    workflow_completed = False
    async for event in workflow_interactive.run(responses=responses, stream=True):
        if event.type == "request_info" and isinstance(event.data, HandoffAgentUserRequest):
            pending_requests.append(event)
        elif event.type == "output":
            if not workflow_completed:
                print(f"\n\u2705 Workflow completed!")
                workflow_completed = True
            if isinstance(event.data, AgentResponseUpdate):
                print(event.data.text or "", end="", flush=True)
            elif event.data:
                print(event.data)

    if exit_requested:
        break

if not pending_requests:
    print("\n\u2705 No more pending requests. Workflow complete.")

print("\n\u2705 Interactive handoff demo complete!")

## 8️⃣ Enable Autonomous Mode for Non-Interactive Workflows

Handoff orchestration is designed for interactive scenarios. However, you can enable **autonomous mode** to allow the workflow to continue without human intervention. When an agent decides not to handoff, the workflow automatically sends a default response.

> **Why is Handoff orchestration inherently interactive?** Unlike other orchestrations where there is only one path to follow after an agent responds, in a Handoff orchestration the agent has the option to either handoff to another agent or continue assisting the user. Because handoffs are achieved through tool calls, if an agent generates a response instead, the workflow delegates back to the user.

| Configuration | Description |
|--------------|-------------|
| `with_autonomous_mode()` | Enable for all agents |
| `with_autonomous_mode(agents=[...])` | Enable for specific agents only |
| `prompts={...}` | Customize the autonomous response message |
| `turn_limits={...}` | Limit autonomous turns per agent |

In [ ]:
print("\n🏦 Running Autonomous Handoff Workflow...")

# Build workflow with autonomous mode enabled
workflow_autonomous = (
    HandoffBuilder(
        name="autonomous_customer_support",
        participants=[triage_agent, refund_agent, order_agent, return_agent],
    )
    .with_start_agent(triage_agent)
    .with_autonomous_mode()
    .build()
)

print("✅ Autonomous workflow built")
print("\n⏳ Running autonomously (no human input required)...\n")

# Run - agents handle routing automatically without human input
workflow_completed = False
async for event in workflow_autonomous.run("Check the status of order ORD-12345", stream=True):
    if event.type == "request_info" and isinstance(event.data, HandoffAgentUserRequest):
        # In autonomous mode, these should be auto-handled, but display if any leak through
        print(f"\n💬 Agent '{event.executor_id}' responded:")
        for msg in event.data.agent_response.messages[-2:]:
            if msg.text:
                print(f"   {msg.author_name or msg.role}: {msg.text}")
    elif event.type == "output":
        if not workflow_completed:
            print(f"\n✅ Autonomous workflow completed!")
            workflow_completed = True
        if isinstance(event.data, AgentResponseUpdate):
            print(event.data.text or "", end="", flush=True)
        elif event.data:
            print(event.data)

## 9️⃣ Customize Autonomous Mode (Subset of Agents, Prompts, Turn Limits)

You can fine-tune autonomous mode with three configuration options:

1. **Subset of agents** — Only specific agents run autonomously
2. **Custom prompts** — Customize the default response message per agent
3. **Turn limits** — Cap autonomous turns to prevent runaway workflows

In [ ]:
# Build workflow with custom routing AND autonomous mode
workflow_variation = (
    HandoffBuilder(
        name="custom_autonomous_support",
        participants=[triage_agent, refund_agent, order_agent, return_agent],
    )
    .with_start_agent(triage_agent)
    .add_handoff(triage_agent, [order_agent, return_agent])
    .add_handoff(return_agent, [refund_agent, triage_agent])
    .add_handoff(order_agent, [triage_agent])
    .add_handoff(refund_agent, [triage_agent])
    .with_autonomous_mode(
        agents=[triage_agent],
        prompts={triage_agent.name: "Continue with your best judgment as the user is unavailable."},
        turn_limits={triage_agent.name: 3},
    )
    .build()
)

print("✅ Custom routing + autonomous mode workflow built")
print("\n⏳ Processing return/refund request...\n")

workflow_completed = False
async for event in workflow_variation.run("I received a damaged item and want to return it for a refund", stream=True):
    if event.type == "request_info" and isinstance(event.data, HandoffAgentUserRequest):
        print(f"\n💬 Agent '{event.executor_id}' says:")
        for msg in event.data.agent_response.messages[-2:]:
            if msg.text:
                print(f"   {msg.author_name or msg.role}: {msg.text}")
    elif event.type == "output":
        if not workflow_completed:
            print(f"\n✅ Custom routing workflow completed!")
            workflow_completed = True
        if isinstance(event.data, AgentResponseUpdate):
            print(event.data.text or "", end="", flush=True)
        elif event.data:
            print(event.data)

## 🔟 Advanced: Define Tools with Approval Required (HITL)

Handoff workflows can include agents with tools that require **human approval** before execution. This is useful for sensitive operations like processing refunds or executing irreversible actions.

Use `@ai_function(approval_mode="always_require")` to mark a tool as requiring approval.

In [ ]:
# Redefine process_refund with approval required
@tool(approval_mode="always_require")
def process_refund_with_approval(order_number: Annotated[str, "Order number to process refund for"]) -> str:
    """Simulated function to process a refund — requires human approval before execution."""
    return f"Refund processed successfully for order {order_number}."

# Recreate agents with the approval-required tool
triage_agent_hitl = Agent(
    client=chat_client,
    instructions=(
        "You are frontline support triage. Route customer issues to the appropriate specialist agents "
        "based on the problem described."
    ),
    description="Triage agent that handles general inquiries.",
    name="triage_agent",
)

refund_agent_hitl = Agent(
    client=chat_client,
    instructions="You process refund requests. Always verify the order number before processing.",
    description="Agent that handles refund requests (requires approval).",
    name="refund_agent",
    tools=[process_refund_with_approval],  # This tool requires human approval
)

order_agent_hitl = Agent(
    client=chat_client,
    instructions="You handle order and shipping inquiries.",
    description="Agent that handles order tracking and shipping issues.",
    name="order_agent",
    tools=[check_order_status],
)

print("✅ Agents with approval-required tools created")
print("   refund_agent now requires human approval for process_refund")

## 1️⃣1️⃣ Handle Both User Input and Tool Approval Requests

When running a handoff workflow with approval-required tools, pending requests can be either:
- **`HandoffAgentUserRequest`** — Agent needs user input to continue
- **`FunctionApprovalRequestContent`** — Agent wants to call a tool that requires human approval

You must check the type of each request and respond appropriately.

In [ ]:
# Build workflow with approval-required tools
workflow_with_approvals = (
    HandoffBuilder(
        name="support_with_approvals",
        participants=[triage_agent_hitl, refund_agent_hitl, order_agent_hitl],
    )
    .with_start_agent(triage_agent_hitl)
    .build()
)

print("✅ Handoff workflow with tool approval built")
print("\n🚀 Starting workflow (will request approval for refund tool)...\n")

# Run the workflow, handling both user input and tool approval requests
pending_requests: list[WorkflowEvent] = []

async for event in workflow_with_approvals.run("My order 12345 arrived damaged. I need a refund.", stream=True):
    if event.type == "request_info":
        pending_requests.append(event)

while pending_requests:
    responses: dict[str, object] = {}

    for request in pending_requests:
        if isinstance(request.data, HandoffAgentUserRequest):
            # Agent needs user input
            print(f"\n💬 Agent '{request.executor_id}' says:")
            for msg in request.data.agent_response.messages[-2:]:
                if msg.text:
                    print(f"   {msg.author_name or msg.role}: {msg.text}")
            print("-" * 40)

            user_input = ""
            while not user_input:
                user_input = input("You (type message): ").strip()
                if not user_input:
                    print("⚠️ Please type a message and press Enter.")
            responses[request.request_id] = HandoffAgentUserRequest.create_response(user_input)

        elif isinstance(request.data, Content) and request.data.type == "function_approval_request":
            # Tool approval request
            func_call = request.data.function_call
            args = func_call.parse_arguments() if hasattr(func_call, 'parse_arguments') else {}

            print(f"\n🔐 Tool approval requested: {func_call.name}")
            print(f"   Arguments: {args}")
            print("-" * 40)

            approval_input = ""
            while approval_input not in ("y", "n"):
                approval_input = input("   Approve? (y/n): ").strip().lower()
                if approval_input not in ("y", "n"):
                    print("⚠️ Please type 'y' or 'n' and press Enter.")
            responses[request.request_id] = request.data.to_function_approval_response(approved=(approval_input == "y"))

    # Send responses and collect new requests
    pending_requests = []
    workflow_completed = False
    async for event in workflow_with_approvals.run(responses=responses, stream=True):
        if event.type == "request_info":
            pending_requests.append(event)
        elif event.type == "output":
            if not workflow_completed:
                print(f"\n✅ Workflow completed!")
                workflow_completed = True
            if isinstance(event.data, AgentResponseUpdate):
                print(event.data.text or "", end="", flush=True)
            elif event.data:
                print(event.data)

print("\n✅ Tool approval handoff demo complete!")

## 1️⃣2️⃣ Implement Checkpointing for Durable Handoff Workflows

For long-running workflows where tool approvals may happen hours or days later, use **checkpointing** to persist workflow state. The workflow can be paused when approval is needed and resumed later from a saved checkpoint.

### Checkpoint Workflow
```
Initial Run → Workflow pauses (approval needed) → Checkpoint saved
     ...time passes...
Resume → Restore checkpoint → Send approval → Workflow completes
```

In [ ]:
from agent_framework import FileCheckpointStorage

# Create checkpoint storage
storage = FileCheckpointStorage(storage_path="./checkpoints")

# Build a durable workflow with checkpoint storage
workflow_durable = (
    HandoffBuilder(
        name="durable_support",
        participants=[triage_agent_hitl, refund_agent_hitl, order_agent_hitl],
        checkpoint_storage=storage,
    )
    .with_start_agent(triage_agent_hitl)
    .build()
)

print("\u2705 Durable handoff workflow with checkpointing built")

# --- Phase 1: Initial run --- workflow pauses when user input is needed ---
print("\n\U0001f4cc Phase 1: Starting workflow (will pause for user input)...")
pending_requests = []
async for event in workflow_durable.run("I need a refund for order 12345", stream=True):
    if event.type == "request_info":
        pending_requests.append(event)

print(f"   {len(pending_requests)} pending request(s) saved. Checkpoint stored automatically.")

# --- Phase 2: Resume from checkpoint and provide input ---
print("\n\U0001f4cc Phase 2: Resuming from checkpoint...")
checkpoints = await storage.list_checkpoints(workflow_name="durable_support")
if checkpoints:
    latest = sorted(checkpoints, key=lambda c: c.timestamp, reverse=True)[0]
    print(f"   Restoring checkpoint: {latest.checkpoint_id}")

    # Restore checkpoint to reload pending requests
    restored_requests = []
    async for event in workflow_durable.run(checkpoint_id=latest.checkpoint_id, stream=True):
        if event.type == "request_info":
            restored_requests.append(event)

    # Send responses
    responses: dict[str, object] = {}
    for req in restored_requests:
        if isinstance(req.data, HandoffAgentUserRequest):
            responses[req.request_id] = HandoffAgentUserRequest.create_response(
                "Yes, please process the refund for order 12345."
            )
            print(f"   \U0001f4dd Responded to agent: {req.executor_id}")

    if responses:
        durable_completed = False
        async for event in workflow_durable.run(responses=responses, stream=True):
            if event.type == "output":
                if not durable_completed:
                    print("\n\u2705 Durable workflow completed after checkpoint restore!")
                    durable_completed = True
                if isinstance(event.data, AgentResponseUpdate):
                    print(event.data.text or "", end="", flush=True)
                elif event.data:
                    print(event.data)
else:
    print("   No checkpoints found.")

## 📝 Key Takeaways

### Handoff Orchestration for Customer Support

| Benefit | Description |
|---------|-------------|
| **Dynamic Routing** | Agents decide which specialist handles the next interaction |
| **Specialized Expertise** | Each agent focuses on their domain |
| **Context Preservation** | Full conversation history maintained across handoffs |
| **Flexible Routing** | Default mesh or custom handoff rules via `add_handoff()` |
| **Autonomous Mode** | Optional mode for unattended workflows with turn limits |
| **Tool Approval (HITL)** | Human approval for sensitive operations |
| **Checkpointing** | Durable workflows that survive process restarts |

### When to Use Handoff vs Other Patterns

| Pattern | When to Use | Key Feature |
|---------|-------------|-------------|
| **Handoff** | Dynamic delegation between peers | Agents transfer full ownership |
| **Sequential** | Linear, dependent steps | Fixed execution order |
| **Magentic** | Complex, adaptive decomposition | Central orchestrator manages plan |
| **Agent-as-Tools** | Primary agent delegates subtasks | Primary retains control |

### Key APIs Reference

| API | Purpose |
|-----|---------|
| `HandoffBuilder` | Build handoff workflows |
| `.with_start_agent()` | Set the entry-point agent |
| `.add_handoff()` | Configure custom routing rules |
| `.with_autonomous_mode()` | Enable autonomous operation |
| `HandoffAgentUserRequest` | Handle user input events |
| `FunctionApprovalRequestContent` | Handle tool approval events |
| `FileCheckpointStorage` | Enable durable checkpointing |
| `workflow.run()` / `workflow.send_responses()` | Non-streaming workflow execution |
| `WorkflowRunResult.get_request_info_events()` | Get pending requests from result |
| `@ai_function(approval_mode="always_require")` | Mark tools requiring approval |

### Production Considerations for FSI

| Consideration | Recommendation |
|---------------|----------------|
| **Routing Rules** | Use `add_handoff()` for controlled routing in regulated environments |
| **Audit Trail** | Log all handoff decisions and agent interactions |
| **Autonomous Mode** | Use sparingly; set `turn_limits` to prevent runaway workflows |
| **Tool Approval** | Use `@ai_function(approval_mode="always_require")` for sensitive operations |
| **Checkpointing** | Enable for long-running workflows that may span sessions |
| **Context Sync** | Remember: only user/agent messages are synchronized, not tool calls |